# Stage 6A: ResNet-34 end-to-end reference
Smoke first; no submission.csv. Gold is development-only. Edit paths in the next cell. Do not infer score from a smoke run.

In [ ]:
S6 = dict(competition='/kaggle/input/competitions/rsna-knee-abnormality-detection',
    weights='', labels='', output='/kaggle/working', resume='',
    size=224, windows=4, batch_size=1, fold=0, seed=42,
    pooling='mean', backbone_lr=1e-5, head_lr=3e-4,
    smoke=False, epochs=12, minutes=480)

S6["implementation_sha256"] = 'fef584b3353c5b757caf1f0b2659d65edec8d76995d69da825368aa942bcd8e0'


In [ ]:
# ============================================================
# v5: Configuration — 288px/130mm 奈奎斯特分辨率 + v5 融合软标签 (teacher-student)
# ============================================================

TARGET_COLUMNS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA',
    'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]
N_CLASSES = len(TARGET_COLUMNS)

# Soft-label columns
PROB_COLS   = [f'prob_{c}' for c in TARGET_COLUMNS]
WEIGHT_COLS = [f'weight_{c}' for c in TARGET_COLUMNS]
MASK_COLS   = [f'mask_{c}' for c in TARGET_COLUMNS]

# ---- 6 Clinical Slots ----
SLOTS = [
    ("SAG_FLUID_FS",   "Sagittal", True,  True),
    ("COR_FLUID_FS",   "Coronal",  True,  True),
    ("AX_FLUID_FS",    "Axial",    True,  True),
    ("SAG_FLUID_NOFS", "Sagittal", True,  False),
    ("COR_T1",         "Coronal",  False, False),
    ("SAG_T1",         "Sagittal", False, False),
]
N_SLOT = len(SLOTS)



In [ ]:
import re
import numpy as np
import pandas as pd
IS_MAIN=True
# ============================================================
# v4: Slot Matching + Laterality Detection + DICOM Header Annotation
# ============================================================

# ---- DICOM Header Annotation (Ref1: annotate_sequences) ----
_SEP = re.compile(r'[_\-.]')
_FATSAT_RX = re.compile(
    r'\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|'
    r'water excit|\btirm\b|\bsting\b|\bfatsup\b'
)
_T1_RX = re.compile(r'\bt1\b|\bt1w\b')
_T2_RX = re.compile(r'\bt2\b|\bt2w\b')
_PD_RX = re.compile(r'\bpd\b|\bpdw\b|proton|\bdp\b|dens')

FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}

_HDR_TAGS = [
    'SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence',
    'RepetitionTime', 'EchoTime', 'Laterality', 'ImageLaterality',
    'ImagePositionPatient', 'PixelSpacing',
]


def _tag_side(group):
    """从 DICOM Laterality 标签推断侧性。"""
    values = [str(x).strip().upper() for x in group.get('Laterality', pd.Series(dtype=object)).dropna()]
    if 'ImageLaterality' in group.columns:
        values += [str(x).strip().upper() for x in group['ImageLaterality'].dropna()]
    values = [x[0] for x in values if x and x[0] in ('L', 'R')]
    return values[0] if values else None


def _position_side(group, min_offset_mm=5.0):
    """从 ImagePositionPatient[0] 推断侧性：DICOM LPS 中 +x = 患者左侧。"""
    xs = []
    for raw in group.get('ImagePositionPatient', pd.Series(dtype=object)).dropna():
        try:
            xs.append(float(str(raw).split('|')[0]))
        except Exception:
            pass
    if not xs:
        return None
    median_x = float(np.median(xs))
    if abs(median_x) < min_offset_mm:
        return None
    return 'R' if median_x < 0 else 'L'


def detect_laterality(headers_df):
    """为每个 study 确定侧性（左/右），结合标签和几何位置。"""
    tagged, positioned = {}, {}
    for study_uid, group in headers_df.groupby('StudyInstanceUID'):
        tagged[study_uid] = _tag_side(group)
        positioned[study_uid] = _position_side(group)

    comparable = [s for s in tagged if tagged[s] and positioned[s]]
    agreement = float(np.mean([
        tagged[s] == positioned[s] for s in comparable
    ])) if comparable else np.nan

    use_position = bool(comparable) and np.isfinite(agreement) and agreement >= 0.85

    resolved = {
        uid: (tagged[uid] or (positioned[uid] if use_position else None))
        for uid in tagged
    }
    coverage = float(np.mean([v is not None for v in resolved.values()]))

    if IS_MAIN:
        print(f'Laterality: tag_coverage={len([v for v in tagged.values() if v])/max(len(tagged),1):.1%}, '
              f'agreement={agreement:.1%} on {len(comparable)} studies, '
              f'final_coverage={coverage:.1%}')
    return resolved


def annotate_sequences(df):
    """从 DICOM header 推断 Fluid/FatSat/Weight，作为 train_series.csv 的 fallback。"""
    df = df.copy()

    # Fat suppression detection
    desc = (df.get('SeriesDescription', '').fillna('') + ' ' +
            df.get('SequenceName', '').fillna(''))
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)

    scan_options = df.get('ScanOptions', '').fillna('').str.upper().str.split('|')
    option_fatsat = scan_options.apply(
        lambda tokens: any(t.strip() in FATSAT_OPTS for t in tokens))
    df['fatsat_detected'] = desc.str.contains(_FATSAT_RX) | option_fatsat

    # Weight detection
    tr = pd.to_numeric(df.get('RepetitionTime', np.nan), errors='coerce')
    te = pd.to_numeric(df.get('EchoTime', np.nan), errors='coerce')
    named_t1 = desc.str.contains(_T1_RX)
    named_t2 = desc.str.contains(_T2_RX)
    named_pd = desc.str.contains(_PD_RX)

    df['weight'] = np.where(
        named_t1 & ~named_t2 & ~named_pd, 'T1',
        np.where(named_t2 & ~named_pd, 'T2',
                 np.where(named_pd, 'PD',
                          np.where(tr < 800, 'T1',
                                   np.where(te > 60, 'T2',
                                            np.where(tr >= 800, 'PD', 'UNK'))))))
    df['fluid_detected'] = df['weight'].isin(['PD', 'T2'])

    return df


# ---- Slot Matching ----
def match_slots_for_study(study_series_df):
    """为单个 study 的每个 slot 匹配最优 series。"""
    slots_found = {}
    for slot_name, plane, fluid, fatsat in SLOTS:
        candidates = study_series_df[
            (study_series_df['Anatomical_Plane'] == plane)
            & (study_series_df['Fluid_Sensitive'] == (1 if fluid else 0))
            & (study_series_df['Fat_Suppression'] == (1 if fatsat else 0))
        ]
        if len(candidates) == 0 and not fluid:
            candidates = study_series_df[
                (study_series_df['Anatomical_Plane'] == plane)
                & (study_series_df['Fluid_Sensitive'] == 0)
            ]
        if len(candidates) > 0:
            best = candidates.sort_values('n_slices', ascending=False).iloc[0]
            slots_found[slot_name] = {
                'series_uid': best['SeriesInstanceUID'],
                'dir': best['dir'],
                'n_slices': int(best['n_slices']),
                'plane': plane,
            }
        else:
            slots_found[slot_name] = None
    return slots_found


def build_study_slot_map(series_meta, dicom_root):
    """为所有 study 构建 slot→series 映射。"""
    df = series_meta.copy()
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    df['SeriesInstanceUID'] = df['SeriesInstanceUID'].astype(str)

    # 计算 DICOM 目录和切片数
    dirs, n_slices_list = [], []
    for _, row in df.iterrows():
        d = str(dicom_root / row['StudyInstanceUID'] / row['SeriesInstanceUID'])
        dirs.append(d)
        if os.path.isdir(d):
            files = [f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))]
            n_dcm = len([f for f in files if f.endswith('.dcm')])
            if n_dcm == 0:
                n_dcm = len([f for f in files if not f.startswith('.')])
            n_slices_list.append(n_dcm)
        else:
            n_slices_list.append(0)
    df['dir'] = dirs
    df['n_slices'] = n_slices_list

    slot_map, study_series_map = {}, {}
    for study_uid, grp in df.groupby('StudyInstanceUID'):
        study_series_map[study_uid] = grp
        slot_map[study_uid] = match_slots_for_study(grp)

    # 统计
    slot_counts = {}
    for slots in slot_map.values():
        for name, sid in slots.items():
            slot_counts[name] = slot_counts.get(name, 0) + (1 if sid is not None else 0)

    if IS_MAIN:
        n_studies = len(slot_map)
        print(f'Slot map: {n_studies} studies')
        for name, count in slot_counts.items():
            print(f'  {name:<18s}: {count:5d}/{n_studies} ({count/n_studies*100:.0f}%)')

    return slot_map, study_series_map

print('Slot matching v4 ready.')


In [ ]:
"""Trainable multi-sequence reference. No network calls or task state at import."""
import hashlib
import numpy as np
import torch
from torch import nn
from torchvision.models import resnet34


def stable_fold(group, folds=5):
    return int(hashlib.sha256(('stage6-v1:' + str(group)).encode()).hexdigest()[:16], 16) % folds


def centers(length, count):
    if length < 3:
        return np.empty(0, dtype=np.int64)
    return np.unique(np.linspace(1, length - 2, min(count, length - 2)).round().astype(np.int64))


class KneeResNet(nn.Module):
    def __init__(self, weights=None, slots=6, classes=12, pooling='mean'):
        super().__init__()
        self.encoder = resnet34(weights=None)
        if weights is not None:
            self.encoder.load_state_dict(weights, strict=True)
        self.encoder.fc = nn.Identity()
        self.pooling = pooling
        self.slot_embedding = nn.Embedding(slots, 512)
        self.attention = nn.Linear(512, classes)
        self.head = nn.Parameter(torch.randn(classes, 512) * .01)
        self.bias = nn.Parameter(torch.zeros(classes))
        self.dropout = nn.Dropout(.2)

    def train(self, mode=True):
        super().train(mode)
        # Small study batches: retain pretrained running statistics, train affine params.
        for module in self.encoder.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()
        return self

    def forward(self, images, valid, slots):
        b, n, c, h, w = images.shape
        valid = valid.bool()
        if not valid.any(1).all():
            raise ValueError('A study has no valid image windows')
        indices = valid.flatten().nonzero().flatten()
        encoded = self.encoder(images.reshape(-1, c, h, w)[indices])
        features = encoded.new_zeros(b * n, 512).index_copy(0, indices, encoded).reshape(b, n, 512)
        features = features + self.slot_embedding(slots)
        scores = self.attention(features).transpose(1, 2)
        if self.pooling == 'mean':
            scores = torch.zeros_like(scores)
        elif self.pooling != 'attention':
            raise ValueError(self.pooling)
        attention = scores.masked_fill(~valid[:, None], -torch.inf).softmax(-1)
        pooled = torch.einsum('bcn,bnd->bcd', attention, features)
        return (self.dropout(pooled) * self.head).sum(-1) + self.bias


def loss_fn(logits, target, weight, mask):
    effective = weight * mask
    if effective.sum() <= 0:
        raise ValueError('Batch has no supervised labels')
    return (nn.functional.binary_cross_entropy_with_logits(logits, target, reduction='none') * effective).sum() / effective.sum()


In [ ]:
"""Stage 6A: offline smoke/pilot; final epoch, no Gold-selected checkpoint."""
import json, time, random, os
from pathlib import Path
import pandas as pd
import pydicom
import cv2
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score


def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(8 << 20), b''):
            h.update(block)
    return h.hexdigest()


def asset(explicit, name):
    if explicit:
        p = Path(explicit)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    matches = list(Path('/kaggle/input').rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f'Configure {name} explicitly: {matches}')
    return matches[0]


def read_volume(directory):
    rows = []
    for p in sorted(directory.iterdir()):
        if not p.is_file():
            continue
        ds = pydicom.dcmread(p)
        orient = np.asarray(ds.ImageOrientationPatient, dtype=float)
        normal = np.cross(orient[:3], orient[3:])
        position = float(np.dot(np.asarray(ds.ImagePositionPatient, dtype=float), normal))
        pixels = ds.pixel_array.astype(np.float32)
        if pixels.ndim != 2:
            raise ValueError(f'Non-2D DICOM: {p}')
        pixels = pixels * float(getattr(ds, 'RescaleSlope', 1)) + float(getattr(ds, 'RescaleIntercept', 0))
        if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
            pixels = -pixels
        rows.append((position, pixels, np.asarray(ds.PixelSpacing, dtype=float)))
    if not rows:
        raise ValueError(f'Empty series: {directory}')
    rows.sort(key=lambda row: row[0])
    spacing = rows[0][2]
    if not np.isfinite(spacing).all() or (spacing <= 0).any():
        raise ValueError(f'Invalid spacing: {directory}')
    volume = np.stack([r[1] for r in rows])
    lo, hi = np.percentile(volume, [1, 99])
    volume = np.clip((volume - lo) / max(hi - lo, 1e-6), 0, 1)
    # Preserve physical aspect ratio and full in-plane coverage; no anatomy crop yet.
    physical = np.array(volume.shape[1:]) * spacing
    shape = np.maximum(1, np.round(physical / physical.max() * S6['size']).astype(int))
    result = np.zeros((len(volume), S6['size'], S6['size']), np.uint8)
    y, x = (S6['size'] - shape) // 2
    for i, image in enumerate(volume):
        result[i, y:y+shape[0], x:x+shape[1]] = (cv2.resize(image, (int(shape[1]), int(shape[0]))) * 255).astype(np.uint8)
    return result


class Studies(Dataset):
    def __init__(self, uids, training=False):
        self.uids, self.training = list(uids), training

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, index):
        uid = self.uids[index]
        filename=hashlib.sha256(uid.encode()).hexdigest()+'.npz'
        path=cache/filename
        if not path.is_file() and S6['resume']:
            path=Path(S6['resume'])/'pixel_cache'/filename
        with np.load(path) as z:
            images, valid = z['images'].copy(), z['valid'].copy()
        images = torch.from_numpy(images).float() / 255
        if self.training:
            images = (images * random.uniform(.9, 1.1)).clamp(0, 1)
        images = (images - torch.tensor([.485,.456,.406])[None,:,None,None]) / torch.tensor([.229,.224,.225])[None,:,None,None]
        row = labels.loc[uid]
        return images, torch.from_numpy(valid), torch.arange(6).repeat_interleave(S6['windows']), torch.tensor(row[PROB_COLS].to_numpy(dtype=np.float32)), torch.tensor(row[WEIGHT_COLS].to_numpy(dtype=np.float32)), torch.tensor(row[MASK_COLS].to_numpy(dtype=np.float32))


def evaluate(uids, name):
    loader = DataLoader(Studies(uids), batch_size=S6['batch_size'], shuffle=False, num_workers=0)
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in loader:
            x, valid, slot = [v.to(device) for v in batch[:3]]
            with torch.autocast('cuda', enabled=device.type == 'cuda'):
                predictions.append(model(x, valid, slot).sigmoid().float().cpu().numpy())
    pred = np.concatenate(predictions)
    pd.DataFrame(pred, index=pd.Index(uids, name='StudyInstanceUID'), columns=TARGET_COLUMNS).to_csv(out / f'{name}_predictions.csv')
    return pred


def run():
    global labels, cache, model, device, out
    started = time.monotonic()
    random.seed(S6['seed']); np.random.seed(S6['seed']); torch.manual_seed(S6['seed'])
    torch.set_num_threads(2)
    out = Path(S6['output']); out.mkdir(parents=True, exist_ok=True)
    cache = out / 'pixel_cache'; cache.mkdir(exist_ok=True)
    root = Path(S6['competition'])
    if not root.is_dir():
        root = Path('/kaggle/input/rsna-knee-abnormality-detection')
    weight = asset(S6['weights'], 'resnet34-b627a593.pth')
    label_path = asset(S6['labels'], 'v5_labels.csv')
    if sha(label_path) != 'c13adffaabf4f8e518abb038282bb1aa09baac7652a9165e030710c457d0be6a':
        raise ValueError('Historical label hash mismatch')
    if not sha(weight).startswith('b627a593'):
        raise ValueError('Official pretrained weight hash mismatch')
    meta = pd.read_csv(root / 'train.csv', dtype={'StudyInstanceUID': str})
    labels = pd.read_csv(label_path, dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    assert labels.index.is_unique
    values = labels[PROB_COLS + WEIGHT_COLS + MASK_COLS].to_numpy()
    assert np.isfinite(values).all() and (values >= 0).all() and (values <= 1).all()
    gold = sorted(meta.loc[meta[TARGET_COLUMNS].notna().all(axis=1), 'StudyInstanceUID'])
    candidates = sorted(set(meta.StudyInstanceUID) - set(gold))
    assert set(candidates).issubset(labels.index), 'Missing labels'
    group_col = 'PatientID' if 'PatientID' in meta and meta.PatientID.notna().all() else 'StudyInstanceUID'
    groups = dict(zip(meta.StudyInstanceUID,meta[group_col].astype(str)))
    gold_groups={groups[u] for u in gold}
    candidates=[u for u in candidates if groups[u] not in gold_groups]
    folds = {u: stable_fold(groups[u]) for u in candidates}
    train = [u for u in candidates if folds[u] != S6['fold']]
    val = [u for u in candidates if folds[u] == S6['fold']]
    if S6['smoke']:
        train, val, gold = train[:8], val[:4], gold[:4]
    assert train and val and not set(train) & set(val)
    assert not {groups[u] for u in train} & {groups[u] for u in val}
    pd.DataFrame([{'StudyInstanceUID': u, 'group': groups[u], 'fold': folds.get(u, -1), 'split': split} for split, ids in [('train', train), ('pseudo_validation', val), ('gold_development', gold)] for u in ids]).to_csv(out / 'split.csv', index=False)
    receipt = dict(config=S6, labels_sha256=sha(label_path), weights_sha256=sha(weight), group_unit=group_col,
                   train_metadata_sha256=sha(root/'train.csv'), series_metadata_sha256=sha(root/'train_series.csv'),
                   gold_independent=False, teacher_oof_provenance_verified=False,
                   status='PREPARING', train_count=len(train), val_count=len(val), gold_count=len(gold))
    resume = Path(S6['resume']) if S6['resume'] else None
    if resume:
        previous=json.loads((resume/'run_receipt.json').read_text())
        for key in ('labels_sha256','weights_sha256','group_unit','train_metadata_sha256','series_metadata_sha256'):
            assert previous[key]==receipt[key], f'Resume mismatch: {key}'
        for key in ('size','windows','batch_size','fold','seed','pooling','backbone_lr','head_lr','smoke','implementation_sha256'):
            assert previous['config'][key]==S6[key], f'Resume mismatch: {key}'
    def report():
        (out / 'run_receipt.json').write_text(json.dumps(receipt, indent=2))
    report()
    series = pd.read_csv(root / 'train_series.csv', dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
    series['dir'] = [str(root/'train_series'/u/s) for u,s in zip(series.StudyInstanceUID,series.SeriesInstanceUID)]
    series['n_slices'] = [len(list(Path(d).iterdir())) if Path(d).is_dir() else 0 for d in series.dir]
    series = series[series.n_slices >= 3]
    coverage = []
    for uid in train + val + gold:
        filename=hashlib.sha256(uid.encode()).hexdigest()+'.npz'
        if resume and (resume/'pixel_cache'/filename).is_file():
            # Read prior immutable cache in place; avoid duplicating ~16 GB.
            with np.load(resume/'pixel_cache'/filename) as z:
                assert z['images'].shape==(6*S6['windows'],3,S6['size'],S6['size'])
                assert z['images'].dtype==np.uint8 and z['valid'].any()
                coverage.append({'uid':uid,'valid_windows':int(z['valid'].sum())})
            continue
        windows = np.zeros((6*S6['windows'], 3, S6['size'], S6['size']), np.uint8)
        valid = np.zeros(len(windows), bool)
        matched = match_slots_for_study(series[series.StudyInstanceUID == uid])
        for slot, (name, *_rest) in enumerate(SLOTS):
            selected = matched[name]
            if selected is None:
                continue
            vol = read_volume(Path(selected['dir']))
            for j, center in enumerate(centers(len(vol), S6['windows'])):
                windows[slot*S6['windows']+j] = vol[center-1:center+2]
                valid[slot*S6['windows']+j] = True
        if not valid.any():
            raise RuntimeError(f'No valid slots for {uid}; fix input contract, do not silently drop study')
        np.savez_compressed(cache / (hashlib.sha256(uid.encode()).hexdigest()+'.npz'), images=windows, valid=valid)
        coverage.append({'uid':uid, 'valid_windows':int(valid.sum())})
        if time.monotonic()-started > S6['minutes']*60:
            receipt.update(status='PAUSED_PREPROCESSING'); report(); return
    pd.DataFrame(coverage).to_csv(out/'coverage.csv', index=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = KneeResNet(torch.load(weight, map_location='cpu', weights_only=True), pooling=S6['pooling']).to(device)
    optimizer = torch.optim.AdamW([{'params':model.encoder.parameters(), 'lr':S6['backbone_lr']},
        {'params':[p for n,p in model.named_parameters() if not n.startswith('encoder.')], 'lr':S6['head_lr']}], weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    loader = DataLoader(Studies(train, True), batch_size=S6['batch_size'], shuffle=True, num_workers=0)
    history=[]
    start_epoch=0
    if resume and (resume/'last.pt').is_file():
        saved=torch.load(resume/'last.pt',map_location=device,weights_only=False)
        model.load_state_dict(saved['model'],strict=True)
        optimizer.load_state_dict(saved['optimizer']); scaler.load_state_dict(saved['scaler'])
        start_epoch=saved['epoch']; history=saved['history']
        random.setstate(saved['python_rng']); np.random.set_state(saved['numpy_rng'])
        torch.set_rng_state(saved['torch_rng'].cpu())
        if device.type=='cuda':
            torch.cuda.set_rng_state_all([v.cpu() for v in saved['cuda_rng']])
    def save_checkpoint(epoch):
        tmp=out/'last.tmp'
        torch.save(dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),
            epoch=epoch,config=S6,receipt=receipt,history=history,python_rng=random.getstate(),
            numpy_rng=np.random.get_state(),torch_rng=torch.get_rng_state(),
            cuda_rng=torch.cuda.get_rng_state_all() if device.type=='cuda' else []),tmp)
        tmp.replace(out/'last.pt')
    save_checkpoint(start_epoch)
    receipt.update(status='TRAINING', device=str(device)); report()
    for epoch in range(start_epoch,S6['epochs']):
        model.train(); total=0.; tick=time.monotonic()
        for batch in loader:
            if time.monotonic()-started>S6['minutes']*60:
                receipt.update(status='PAUSED_TRAINING',completed_epochs=epoch,
                    note='Resume from last complete epoch; partial epoch discarded'); report(); return
            x,valid,slot,y,w,m = [v.to(device) for v in batch]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=device.type == 'cuda'):
                loss=loss_fn(model(x,valid,slot), y,w,m)
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite loss')
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            if 'first_backbone_gradient_norm' not in receipt:
                grad=model.encoder.conv1.weight.grad
                assert grad is not None and torch.isfinite(grad).all() and grad.abs().sum()>0
                receipt['first_backbone_gradient_norm']=float(grad.norm())
                report()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
            scaler.step(optimizer); scaler.update(); total += loss.item()
        history.append(dict(epoch=epoch+1, loss=total/len(loader), seconds=time.monotonic()-tick))
        if epoch==0 or (epoch+1)%4==0:
            validation=evaluate(val,'pseudo_validation')
            history[-1]['pseudo_validation_mse']=float(np.mean((validation-labels.loc[val,PROB_COLS].to_numpy(float))**2))
        print(history[-1], flush=True)
        save_checkpoint(epoch+1)
        (out/'history.json').write_text(json.dumps(history, indent=2))
        if time.monotonic()-started > S6['minutes']*60:
            receipt.update(status='PAUSED_TRAINING', completed_epochs=epoch+1); report(); return
    pred=evaluate(val, 'pseudo_validation')
    truth=labels.loc[val, PROB_COLS].to_numpy(float)
    receipt['pseudo_validation_mse']=float(np.mean((pred-truth)**2))
    pred=evaluate(gold, 'gold_development')
    truth=meta.set_index('StudyInstanceUID').loc[gold,TARGET_COLUMNS].to_numpy(float)
    auc=[float(roc_auc_score(truth[:,i], pred[:,i])) if len(np.unique(truth[:,i]))==2 else None for i in range(12)]
    receipt.update(status='SMOKE_COMPLETE' if S6['smoke'] else 'PILOT_COMPLETE', gold_auc_by_class=dict(zip(TARGET_COLUMNS,auc)),
                   completed_epochs=S6['epochs'], seconds=time.monotonic()-started)
    report()


run()
